# Main household language in northern Nigeria

**131 Local Government Areas across six states — maps from REACH/ClearGlobal data**

This notebook prepares the data for four thematic maps produced in QGIS: survey
coverage, the prevalent household language by LGA, the spread of Hausa, and the
language composition of the three north-eastern states.

The analysis deliberately focuses on the **limitations of the data**. The source
merges two humanitarian surveys built on different instruments, and that shapes
what the maps can and cannot legitimately say.

---

## Sources

**Dataset** — ClearGlobal, *Language use in Nigeria (admin 2)*, on HDX
<https://data.humdata.org/dataset/nigeria-languages>

**Administrative boundaries** — OCHA, *Nigeria administrative level 0–3 boundaries (COD-AB)*
<https://data.humdata.org/dataset/cod-ab-nga>

**Source surveys** — two Multi-Sector Needs Assessments by REACH / IMPACT Initiatives:

- **North-east**, collected 2 August – 2 October 2021, 60 LGAs in Adamawa, Borno
  and Yobe, 8,745 household questionnaires
  <https://repository.impact-initiatives.org/document/impact/0a02ead7/REACH-NGA-Dataset-and-Analysis-North-East-Nigeria-MSNA-2021-November-2021.xlsx>
- **North-west**, collected 1 March – 8 July 2022, 71 LGAs in Katsina, Sokoto and
  Zamfara, 11,090 household questionnaires
  <https://repository.impact-initiatives.org/document/impact/c011a778/REACH-NGA2105-NW-MSNA-Datasets-Analysis.xlsx>

## 0. What `proportion_value` measures

Before any analysis, the definition of the variable — because it determines what
can legitimately be concluded.

In both questionnaires the underlying question is the same:

> *What is the main language your household uses at home?*

(north-west: variable `main_language_home`; north-east: variable
`questionnaire_aap_communication_hh_main_language_spoken`)

Three decisive properties:

- **Single response** — one language per household, not a list. The data does not
  measure multilingualism.
- **Household unit**, not individual.
- **Main language at home**, not languages known or used outside the home.

So `proportion_value` is the **share of households reporting that language as
their main one at home**. It is not the share of speakers.

The distinction is not cosmetic. Hausa is widely used as a lingua franca across
northern Nigeria by people who speak something else at home: actual speakers are
considerably more numerous than these maps suggest.

Because each household picks exactly one language, proportions sum to 1 within
each LGA — verified further down.

**Correct map label:** *Main household language — share of households (%)*

## 1. Loading and structure

In [2]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# All paths are relative to the notebook folder, so the project
# stays runnable on any machine.
FOLDER = Path.cwd()
OUT = FOLDER / "output"
OUT.mkdir(exist_ok=True)

df = pd.read_csv(FOLDER / "clearglobal_language_use_nga_admin2.csv")
df.head()

,location_code,location_name,location_level,language_code,language_name,language_rank,proportion_value,reliability_score,dataset_name,url,source,datetime_published,date_creation,representivity_rating
0,NG036003,Damaturu,2,bura1272,Bura,7,0.012882,0.7917,REACH NGA Dataset and Analysis North East Nige...,https://repository.impact-initiatives.org/docu...,form,03-23-2022,07-08-2025 16:29:31,high
1,NG036003,Damaturu,2,bade1248,Bade,5,0.025765,0.7917,REACH NGA Dataset and Analysis North East Nige...,https://repository.impact-initiatives.org/docu...,form,03-23-2022,07-08-2025 16:29:31,high
2,NG036003,Damaturu,2,kare1348,Karekare,4,0.042548,0.7917,REACH NGA Dataset and Analysis North East Nige...,https://repository.impact-initiatives.org/docu...,form,03-23-2022,07-08-2025 16:29:31,high
3,NG036003,Damaturu,2,ngam1282,Ngamo,9,0.000049,0.7917,REACH NGA Dataset and Analysis North East Nige...,https://repository.impact-initiatives.org/docu...,form,03-23-2022,07-08-2025 16:29:31,high
4,NG036003,Damaturu,2,haus1257,Hausa,1,0.427755,0.7917,REACH NGA Dataset and Analysis North East Nige...,https://repository.impact-initiatives.org/docu...,form,03-23-2022,07-08-2025 16:29:31,high


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 674 entries, 0 to 673
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   location_code          674 non-null    str    
 1   location_name          674 non-null    str    
 2   location_level         674 non-null    int64  
 3   language_code          674 non-null    str    
 4   language_name          674 non-null    str    
 5   language_rank          674 non-null    int64  
 6   proportion_value       674 non-null    float64
 7   reliability_score      674 non-null    float64
 8   dataset_name           674 non-null    str    
 9   url                    674 non-null    str    
 10  source                 674 non-null    str    
 11  datetime_published     674 non-null    str    
 12  date_creation          674 non-null    str    
 13  representivity_rating  674 non-null    str    
dtypes: float64(2), int64(2), str(10)
memory usage: 73.8 KB


In [4]:
print(f"rows:               {len(df)}")
print(f"distinct LGAs:      {df.location_code.nunique()}")
print(f"distinct languages: {df.language_name.nunique()}")
print(f"states covered:     {df.location_code.str[:5].nunique()}")
print()
print("languages recorded per LGA:")
print(df.location_code.value_counts().describe())

rows:               674
distinct LGAs:      131
distinct languages: 91
states covered:     6

languages recorded per LGA:
count    131.000000
mean       5.145038
std        4.716782
min        1.000000
25%        2.000000
50%        3.000000
75%        6.000000
max       25.000000
Name: count, dtype: float64


The dataset is in **long format**: one row per LGA–language pair. A shapefile has
one polygon per LGA, so every join will need a table with exactly one row per LGA.

## 2. Geographic coverage

`location_code` follows the pattern `NG` + three state digits + three LGA digits,
numbered alphabetically within each state.

In [5]:
STATES = {"NG002": "Adamawa", "NG008": "Borno",   "NG021": "Katsina",
          "NG034": "Sokoto",  "NG036": "Yobe",    "NG037": "Zamfara"}

# Total LGAs per state under the Nigerian administrative division
LGA_TOTAL = {"Adamawa": 21, "Borno": 27, "Katsina": 34,
             "Sokoto": 23, "Yobe": 17, "Zamfara": 14}

lga = (df[["location_code", "location_name"]]
       .drop_duplicates()
       .sort_values("location_code")
       .reset_index(drop=True))
lga["state"] = lga.location_code.str[:5].map(STATES)

cov = (lga.groupby("state").size().rename("in dataset").to_frame()
         .assign(**{"total": lambda d: d.index.map(LGA_TOTAL)}))
cov["missing"] = cov["total"] - cov["in dataset"]
print(cov)
print(f"\ntotal: {len(lga)} LGAs out of 774 nationwide")

         in dataset  total  missing
state                              
Adamawa          21     21        0
Borno            22     27        5
Katsina          34     34        0
Sokoto           23     23        0
Yobe             17     17        0
Zamfara          14     14        0

total: 131 LGAs out of 774 nationwide


Six states out of 36, three in each of the two northern geopolitical zones:

- **North-west** — Katsina, Sokoto, Zamfara (Jigawa, Kaduna, Kano and Kebbi are absent)
- **North-east** — Adamawa, Borno, Yobe (Bauchi, Gombe and Taraba are absent)

The selection is not geographic but **humanitarian**. MSNAs survey where a response
operation is active: Adamawa, Borno and Yobe are the *BAY states* of the Lake Chad
basin crisis; Katsina, Sokoto and Zamfara are the states worst affected by armed
banditry in the north-west.

This needs stating plainly: **the dataset does not describe "northern Nigeria" but
the crisis-affected areas of the north.** Kano in particular is missing — the most
populous northern state and the historic centre of the Hausa language — as is
Kaduna, the main zone of linguistic transition towards the Middle Belt.

### The five missing Borno LGAs

Borno appears with 22 LGAs out of 27. The absent codes are `NG008001`, `NG008010`,
`NG008017`, `NG008022` and `NG008026`. Since numbering is alphabetical and all 22
present codes match the official list exactly, the missing ones are **Abadam,
Guzamala, Kukawa, Marte and Nganzai** — all in the north of the state, in the Lake
Chad basin.

The north-east survey README confirms it: data collection covered *"accessible
areas from 60 LGAs"*, excluding areas deemed physically inaccessible. This is not a
flaw in the file but a limit of fieldwork.

**Consequence for the maps:** those five LGAs must be shown as *no data*, never as
*language absent*.

In [6]:
expected = {f"NG008{n:03d}" for n in range(1, 28)}
present = set(lga.loc[lga.state == "Borno", "location_code"])
print("missing Borno codes:", sorted(expected - present))

missing Borno codes: ['NG008001', 'NG008010', 'NG008017', 'NG008022', 'NG008026']


In [7]:
with pd.option_context("display.max_rows", None):
    display(lga)

,location_code,location_name,state
0,NG002001,Demsa,Adamawa
1,NG002002,Fufore,Adamawa
2,NG002003,Ganye,Adamawa
3,NG002004,Gombi,Adamawa
4,NG002005,Girei,Adamawa
5,NG002006,Guyuk,Adamawa
6,NG002007,Hong,Adamawa
7,NG002008,Jada,Adamawa
8,NG002009,Lamurde,Adamawa
9,NG002010,Madagali,Adamawa


## 3. The two source surveys

The file is not a single survey: it merges two rounds run in different years, over
different areas, with no geographic overlap.

In [8]:
print(df.dataset_name.value_counts(), "\n")
print(df.representivity_rating.value_counts(), "\n")
print(df.reliability_score.value_counts())

dataset_name
REACH NGA Dataset and Analysis North East Nigeria MSNA 2021 November 2021    513
REACH Northwest Nigeria MSNA 2022 Data & Analysis                            161
Name: count, dtype: int64 

representivity_rating
high        513
moderate    161
Name: count, dtype: int64 

reliability_score
0.79170    513
0.65262    161
Name: count, dtype: int64


In [9]:
df["source"] = df.dataset_name.str.contains("North East").map({True: "NE 2021", False: "NW 2022"})
print(pd.crosstab(df.location_code.str[:5].map(STATES), df.source))

source         NE 2021  NW 2022
location_code                  
Adamawa            261        0
Borno              147        0
Katsina              0       70
Sokoto               0       61
Yobe               105        0
Zamfara              0       30


The three metadata columns line up and identify two clean blocks:

| | North-east 2021 | North-west 2022 |
|---|---|---|
| rows | 513 | 161 |
| `representivity_rating` | high | moderate |
| `reliability_score` | 0.7917 | 0.65262 |
| states | Adamawa, Borno, Yobe | Katsina, Sokoto, Zamfara |

`reliability_score` takes **only two values**: it is a score per source, not per
LGA. Mapping it as a continuous variable would be misleading.

### The core problem: the answer lists are not comparable

The two surveys ask the **same question** but offer very different answer lists.

The north-west questionnaire (sheet `The Tool - Choices`, list `lang_home`) has
**six options**: English, Hausa, Fulfulde, Yoruba, Igbo, Other. No local minority
language is named.

The north-east questionnaire records **28 distinct responses**, including bura,
karekare, marghi, kilba, bachama, chibok, kanakuru, higgi, mandara and glavda.

In [10]:
diversity = (df.groupby("source")
               .agg(rows=("language_name", "size"),
                    LGAs=("location_code", "nunique"),
                    distinct_languages=("language_name", "nunique")))
diversity["languages_per_LGA"] = (diversity.rows / diversity.LGAs).round(1)
print(diversity)

         rows  LGAs  distinct_languages  languages_per_LGA
source                                                    
NE 2021   513    60                  85                8.6
NW 2022   161    71                  12                2.3


In [11]:
nw = df[df.source == "NW 2022"].language_name.value_counts()
print("Languages recorded in the north-west:\n")
print(nw)

Languages recorded in the north-west:

language_name
Hausa             71
Fula              44
English           23
Yoruba             9
Unknown            5
Igbo               2
Igala              2
Arabic             1
Zarma              1
Central Kanuri     1
Nupe-Nupe-Tako     1
Izora              1
Name: count, dtype: int64


**This is not a sampling effect.** The north-west interviewed *more* households
than the north-east — 11,090 against 8,745 — and recorded 12 distinct languages
against 85. With a larger sample it captured seven times less diversity.

The cause is therefore the instrument, not sample size. The few north-western
entries beyond the six questionnaire options (Igala, Zarma, Nupe, Izora) appear
once or twice and come from the free-text *Other* field.

**Direct consequence for the maps.** A household in Katsina or Zamfara speaking a
minority language at home had no way to report it: it either chose *Other* or gave
Hausa as an approximation. The Hausa share in the north-west is therefore
**likely overstated**, by an amount these data cannot quantify.

The visual contrast between a compact north-west and a mosaic north-east reflects
instrument design as much as linguistic reality. **Comparisons between the two
regions should be avoided; comparisons within each region remain valid.**

## 4. Quality checks

The dataset is published as clean, but the checks are worth running anyway.

In [12]:
nulls = df.isna().sum()
nulls = nulls[nulls > 0]

print("null values:")
print(nulls.to_string() if not nulls.empty else "none")

print("\nduplicate LGA+language pairs:",
      df.duplicated(["location_code", "language_code"]).sum())

# Since this is a single-response question, proportions
# must sum to 1 within each LGA.
sums = df.groupby("location_code")["proportion_value"].sum()
print("\nsum of proportions per LGA — min/max:",
      round(sums.min(), 6), round(sums.max(), 6))
print("LGAs not summing to 1 (tolerance 1e-6):",
      (sums.sub(1).abs() > 1e-6).sum())

null values:
none

duplicate LGA+language pairs: 0

sum of proportions per LGA — min/max: 1.0 1.0
LGAs not summing to 1 (tolerance 1e-6): 0


The sums of 1 confirm what was stated at the top: one response per household,
shares of a total. A multiple-response question could not have produced this.

## 5. Map 1 — Survey coverage

Before showing any value, show **where the data exists**. The COD-AB shapefile has
774 polygons; the dataset covers 131.

The CSV exported below holds only code and name: it acts as a presence marker for
the QGIS join.

In [13]:
lga_export = lga[["location_code", "location_name", "state"]]

assert len(lga_export) == 131
assert lga_export.location_code.duplicated().sum() == 0

lga_export.to_csv(OUT / "lga_coverage.csv", index=False, encoding="utf-8")
print(f"saved: {OUT / 'lga_coverage.csv'}  ({len(lga_export)} rows)")

saved: c:\Users\Simona\Desktop\Nigeria\Prove_GIS_Lingue\output\lga_coverage.csv  (131 rows)


**QGIS procedure**

1. Load `nga_admin2.shp` (LGA level of COD-AB).
2. Load `lga_coverage.csv` as delimited text, UTF-8, *no geometry*.
3. Join: `location_code` ↔ `adm2_pcode`, empty prefix.
4. Check: `"location_name" IS NOT NULL` must select 131 features.
5. Symbology *rule-based*: `IS NOT NULL` → solid fill; `ELSE` → grey, labelled
   *Not surveyed*.

<img src="Images/LGA_with_data.png" width="700">

*LGAs covered by the two REACH MSNA surveys (n = 131 of 774).*

## 6. Map 2 — Prevalent household language by LGA

`language_rank == 1` selects, for each LGA, the language reported as their main one
by the largest number of households.

In [14]:
prevalent = (
    df[df.language_rank == 1]
    .loc[:, ["location_code", "location_name", "language_name",
             "proportion_value", "reliability_score"]]
    .rename(columns={"language_name": "prevalent_language",
                     "proportion_value": "prevalent_share"})
    .reset_index(drop=True)
)

assert len(prevalent) == 131
assert prevalent.location_code.duplicated().sum() == 0

print("distinct prevalent languages:", prevalent.prevalent_language.nunique(), "\n")
print(prevalent.prevalent_language.value_counts(), "\n")
print(prevalent.prevalent_share.describe())

prevalent.to_csv(OUT / "prevalent_language.csv", index=False, encoding="utf-8")

distinct prevalent languages: 11 

prevalent_language
Hausa                91
Central Kanuri       18
Eastern Fula          9
Bura                  4
Bacama                3
Kilba-South Margi     1
Longuda               1
Cibak                 1
Marghic               1
Karekare              1
Chamba Donga          1
Name: count, dtype: int64 

count    131.000000
mean       0.790283
std        0.229743
min        0.215321
25%        0.544697
50%        0.918918
75%        0.990591
max        1.000000
Name: prevalent_share, dtype: float64


Eleven categories, very unevenly distributed: Hausa prevails in 91 of 131 LGAs,
while six languages appear only once — three in Adamawa (Kilba-South Margi,
Longuda, Chamba Donga), two in Borno (Cibak, Marghic) and one in Yobe (Karekare).

**A limitation to keep in mind.** A categorical map hides *how strongly* the
language prevails. With prevalent shares ranging from 0.215 to 1.000, an LGA where
the top language is reported by 21% of households looks identical to one where it
reaches 100%. Kala/Balge, with Kanuri at 51% and Shuwa Arabic at 49%, is
sociolinguistically the opposite of a monolingual LGA, yet carries the same solid
colour.

One way to address this in QGIS is a data-defined override on fill opacity, scaled
to `prevalent_share`. It is not applied in the map below, which uses solid colours.

**Taxonomic note.** The categories include *Marghic* and *Kilba-South Margi*, which
are not single languages but groupings of varieties. The dataset mixes levels of
granularity: the categories are not all comparable with one another.

<img src="Images/Most_reported_language.png" width="700">

*Most frequently reported main household language, by LGA.*

## 7. Map 3 — Spread of Hausa

The function below extracts any single language and maps it back onto the full list
of 131 LGAs, so it always returns one row per LGA even where the language does not
appear.

The `present` column distinguishes a **genuine absence** (language not recorded in
that LGA) from a **measured zero** — a distinction that `fillna(0)` alone would
erase.

In [15]:
def language_extent(df, language):
    # One row per LGA with the share of the requested language (0 where absent).
    if language not in df["language_name"].values:
        raise ValueError(f"'{language}' does not appear in the dataset")

    base = df[["location_code", "location_name"]].drop_duplicates()

    sub = (df.loc[df["language_name"] == language,
                  ["location_code", "proportion_value",
                   "reliability_score", "language_rank"]]
             .rename(columns={"proportion_value": "share",
                              "reliability_score": "reliability",
                              "language_rank": "rank"}))

    out = base.merge(sub, on="location_code", how="left")
    out["present"] = out["share"].notna()
    out["share"] = out["share"].fillna(0)
    return out


def export_language(df, language, folder=OUT):
    d = language_extent(df, language)
    name = language.lower().replace(" ", "_").replace("/", "_")
    d.to_csv(folder / f"{name}_lga.csv", index=False, encoding="utf-8")
    print(f"{language}: present in {d.present.sum()} of {len(d)} LGAs "
          f"— saved to {name}_lga.csv")
    return d

In [16]:
hausa = export_language(df, "Hausa")
print()
print(hausa.share.describe())
print()
print("rank of Hausa in the LGAs where it is present:")
print(hausa["rank"].value_counts().sort_index())

Hausa: present in 130 of 131 LGAs — saved to hausa_lga.csv

count    131.000000
mean       0.669536
std        0.357871
min        0.000000
25%        0.346379
50%        0.880701
75%        0.990591
max        1.000000
Name: share, dtype: float64

rank of Hausa in the LGAs where it is present:
rank
1.0    91
2.0    27
3.0    11
4.0     1
Name: count, dtype: int64


Hausa is present in 130 of 131 LGAs but **prevalent in only 91**: it is the second
language in 27 LGAs, third in 11 and fourth in 1. A gradient map does not
distinguish these situations — an LGA at 40% where Hausa ranks first tells a
different story from one at 40% where it ranks second behind Kanuri.

`reliability` takes only the two values seen earlier (0.7917 and 0.65262): it
identifies the source survey, not a property of the individual LGA.

### The single LGA without Hausa

An isolated value can be a real finding or a spelling mismatch. It has to be
checked, not assumed.

In [17]:
print(hausa.loc[~hausa.present, ["location_code", "location_name"]], "\n")
print("name variants in the dataset:",
      [l for l in df.language_name.unique() if "aus" in l.lower()])

   location_code location_name
58      NG008015    Kala/Balge 

name variants in the dataset: ['Hausa', 'Hausa Sign Language']


In [18]:
print(df[df.location_code == "NG008015"]
      [["language_name", "language_rank", "proportion_value", "reliability_score"]]
      .sort_values("language_rank")
      .to_string(index=False))

 language_name  language_rank  proportion_value  reliability_score
Central Kanuri              1          0.510941             0.7917
Chadian Arabic              2          0.486483             0.7917
         Afade              3          0.002577             0.7917


Both checks come back clean.

`Hausa Sign Language` is a distinct language, not a spelling variant: exact-match
filtering correctly excludes it.

**Kala/Balge** (Borno, on the Cameroon border) is surveyed normally, with three
languages summing to 1: Central Kanuri 0.511, Chadian Arabic 0.486, Afade 0.003.
Its `reliability_score` is in line with other LGAs. A Kanuri and Arabic-speaking
profile without Hausa is consistent with its position: this is a genuine absence,
not a gap in the data.

*Terminological note:* the north-east questionnaire option was `shuwa arabic`;
ClearGlobal normalises it to *Chadian Arabic*. Same glottocode.

### Colour scale classification

The distribution is heavily skewed — median 0.88, third quartile 0.99 — so equal
intervals would flatten the map: half the LGAs would fall into the same band and
the informative variation, between 0.2 and 0.6, would disappear.

**User-defined breaks** were used instead, chosen to be interpretable and to stay
identical when other languages are mapped, so that maps remain comparable:

| Class | Reading |
|---|---|
| 0 – 0.25 | marginal |
| 0.25 – 0.50 | minority |
| 0.50 – 0.75 | around half |
| 0.75 – 0.95 | majority |
| 0.95 – 1.00 | near-universal |

QGIS uses right-closed intervals, so a value equal to a break falls in the lower
class. Shares carry many decimals and none lands exactly on a break.

<img src="Images/Hausa.png" width="700">

*Share of households reporting Hausa as their main household language, by LGA.*

## 8. Map 4 — Language composition of the three north-eastern states

The three previous maps work at LGA level. This one aggregates to **state** level
and shows composition as one pie chart per state — as in the first version of this
map, which however contained an aggregation error, corrected here.

**Why the north-east only.** The north-western questionnaire offered six answer
options, so pie charts for Katsina, Sokoto and Zamfara would all be 95–98% Hausa
with no other slice above 5%: three nearly identical circles, communicating less
than a sentence would. The north-east has three clearly distinct profiles.

### The error to avoid: aggregating from LGA to state

The dataset gives a proportion per **LGA–language pair**. Taking it to state level
requires an average, and the denominator makes all the difference.

| Method | Denominator | Result |
|---|---|---|
| **wrong** | only the LGAs where the language appears | slices do not add up |
| **correct** | all surveyed LGAs in the state | sums to exactly 100% |

The first method inflates every language in proportion to how localised it is: a
language present in 2 of 22 LGAs with a share of 0.8 would come out as 80% of the
state instead of 7.3%. In the first version of this map slices summed to **102.5%
for Zamfara** and **206.3% for Borno** — an unmistakable symptom of the wrong
denominator.

In the code below the correct denominator is `n_lga`, the number of surveyed LGAs
in each state, and the closing `assert` checks that every row sums to 100.

In [19]:
NE = {"NG002": "Adamawa", "NG008": "Borno", "NG036": "Yobe"}
SLICES = ["Hausa", "Central Kanuri", "Fulfulde", "Bura", "Chamba Donga"]

ne = df[df.location_code.str[:5].isin(NE)].copy()
ne["adm1_pcode"] = ne.location_code.str[:5]
ne["adm1_name"] = ne.adm1_pcode.map(NE)
# the two surveys name the same language differently
ne["language"] = ne.language_name.replace({"Eastern Fula": "Fulfulde"})

n_lga = ne.groupby("adm1_name").location_code.nunique()

# LGA -> state aggregation: mean over ALL surveyed LGAs in the state.
# Dividing by only the LGAs where the language appears inflates the
# values and the slices no longer sum to 100.
shares = (ne.groupby(["adm1_name", "language"]).proportion_value.sum()
            .unstack(fill_value=0)
            .div(n_lga, axis=0))

pies = shares[SLICES].copy()
pies["Other"] = 1 - pies.sum(axis=1)
pies = (pies * 100).round(4)

pies.insert(0, "adm1_pcode", pies.index.map({v: k for k, v in NE.items()}))
pies.insert(1, "n_lga", n_lga)
pies.insert(2, "n_languages", ne.groupby("adm1_name").language.nunique())
pies = pies.reset_index()
pies.columns = [x.replace(" ", "_") for x in pies.columns]

_cols = [x.replace(" ", "_") for x in SLICES] + ["Other"]
_gap = (pies[_cols].sum(axis=1) - 100).abs().max()
assert _gap < 0.01, f"slices do not sum to 100 (max gap {_gap:.4f})"

print(pies.shape)

(3, 10)


In [20]:
# --- Labels for the callout boxes next to each pie ---
ABBR = {"Central_Kanuri": "C. Kanuri", "Chamba_Donga": "Chamba Donga"}
THRESHOLD = 0.5        # entries below this % are omitted from the box

# how many languages fall into "Other", per state
_named = [x.replace("_", " ") for x in SLICES]
n_other = (ne.groupby("adm1_name").language.nunique()
           - ne[ne.language.isin(_named)].groupby("adm1_name").language.nunique())

def _label(r):
    items = [(ABBR.get(x, x), r[x]) for x in
             [s.replace(" ", "_") for s in SLICES] if r[x] >= THRESHOLD]
    items.sort(key=lambda t: -t[1])
    rows = [f"{n}: {q:.1f}%" for n, q in items]
    # "Other" is not a language: it stays last regardless of size,
    # with the number of languages it aggregates in brackets
    rows.append(f"Other ({int(n_other[r.adm1_name])}): {r['Other']:.1f}%")
    return " | ".join(rows)

pies["label"] = pies.apply(_label, axis=1)
pies.to_csv(OUT / "pies_northeast.csv", index=False, encoding="utf-8")

for _, r in pies.iterrows():
    print(r.adm1_name); print("  " + r.label.replace(" | ", "\n  ")); print()

Adamawa
  Hausa: 31.1%
  Fulfulde: 26.4%
  Chamba Donga: 6.6%
  Bura: 1.0%
  C. Kanuri: 0.9%
  Other (61): 34.1%

Borno
  C. Kanuri: 43.4%
  Hausa: 19.1%
  Bura: 13.6%
  Fulfulde: 3.9%
  Other (26): 20.1%

Yobe
  Hausa: 49.3%
  C. Kanuri: 25.6%
  Fulfulde: 11.2%
  Bura: 1.5%
  Other (22): 12.3%



**QGIS procedure**

1. Load `nga_admin1.shp` (state level of COD-AB) — not the GADM version, whose
   codes are incompatible.
2. Load `pies_northeast.csv` as delimited text, UTF-8, *no geometry*.
3. Join: `adm1_pcode` ↔ `adm1_pcode`, empty prefix.
4. **Diagrams → Pie chart**: assign the six slice columns in decreasing order of
   size, with `Other` in neutral grey. Fixed size, not scaled: varying it by area
   or population would encode two variables in one symbol.
5. **Rendering → Show diagram**, data-defined override `"Hausa" IS NOT NULL`, so
   pies appear only on the three surveyed states.
6. **Labels → Single labels**, expression:
   `upper("adm1_name") || '\n\n' || replace("label", ' | ', '\n')`
   with a white background, *Around point* placement, and **Callouts** enabled to
   connect each box to its pie.

*Note:* field names must be adapted to the prefix actually used by the join.

<img src="Images/North-East_Languages_Distribution.png" width="700">

*Main household language in Adamawa, Borno and Yobe. Share of households, averaged
across the 60 surveyed LGAs (REACH MSNA 2021). "Other" aggregates all languages
outside the five largest, with the number of languages in brackets.*